In [176]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

drop_cols = ["PassengerId", "Cabin", "Ticket"]
train_df.drop(drop_cols, axis=1, inplace=True)
test_df.drop(drop_cols, axis=1, inplace=True)

for df in [train_df, test_df]:
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["Title"] = df["Name"].str.extract(r"([A-Za-z]+)\.", expand=False)

train_df.drop(["Name", "SibSp", "Parch"], axis=1, inplace=True)
test_df.drop(["Name", "SibSp", "Parch"], axis=1, inplace=True)

age_median = train_df["Age"].median()
embarked_mode = train_df["Embarked"].mode()[0]

train_df["Age"] = train_df["Age"].fillna(age_median)
test_df["Age"] = test_df["Age"].fillna(age_median)

train_df["Embarked"] = train_df["Embarked"].fillna(embarked_mode)
test_df["Embarked"] = test_df["Embarked"].fillna(embarked_mode)

hot_cols = ["Sex", "Embarked", "Title"]
train_df = pd.get_dummies(train_df, columns=hot_cols)
test_df = pd.get_dummies(test_df, columns=hot_cols)

test_df = test_df.reindex(columns=train_df.drop("Survived", axis=1).columns, fill_value=0)

X = train_df.drop("Survived", axis=1)
y = train_df["Survived"]

In [179]:
# training the model 

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.ensemble import RandomForestClassifier


clf =  RandomForestClassifier(random_state=42, n_estimators=100, max_depth=6)
clf.fit(X_train, y_train)

predict = clf.predict(X_test)
train_acc = clf.score(X_train,y_train)
test_acc = clf.score(X_test, y_test)

print("train acc: ", train_acc)
print("test acc: ", test_acc)


test_predict = clf.predict(test_df)
# test_predict

submission = pd.DataFrame({
    "PassengerId": pd.read_csv('../data/test.csv')["PassengerId"],
    "Survived": test_predict
})

submission.to_csv("../data/submission.csv", index=False)

train acc:  0.8679775280898876
test acc:  0.8324022346368715
